# Vector Search — Hands-On

**LLM Engineering · Domain 1 · Roadmap Weeks 09/14**

Companion to `02 Literature Notes/LLM Engineering/Vector Search` and the deck
`Lesson_05_Vector_Search.pptx`. Runs offline; the FAISS section is optional and
degrades gracefully if faiss isn't installed.

**Sources:** Pinecone vector DB guide; FAISS (arXiv:1702.08734); Qdrant/Weaviate/Chroma docs.

## 0. Setup — a synthetic normalized corpus

In [ ]:
%pip install -q numpy
import numpy as np
rng = np.random.RandomState(0)
N, d = 20_000, 128
corpus = rng.randn(N, d).astype("float32")
corpus /= np.linalg.norm(corpus, axis=1, keepdims=True)
# a query near vector #123
q = corpus[123] + 0.05 * rng.randn(d).astype("float32"); q /= np.linalg.norm(q)
print("corpus:", corpus.shape)

## 1. Exact search (the baseline you always keep)

In [ ]:
def flat_topk(query, matrix, k=10):
    scores = matrix @ query
    idx = np.argpartition(-scores, k)[:k]
    return idx[np.argsort(-scores[idx])]

exact = flat_topk(q, corpus, k=10)
print("exact top-10 ids:", exact.tolist())
print("nearest is #123? ->", 123 in exact.tolist())

## 2. Metric equivalence: dot == cosine after normalization

In [ ]:
a, b = corpus[10], corpus[20]
cos = (a @ b) / (np.linalg.norm(a)*np.linalg.norm(b))
print("cosine    :", round(float(cos), 4))
print("dot (norm):", round(float(a @ b), 4))
print("2-2*dot   :", round(2 - 2*float(a@b), 4), " ||a-b||^2:", round(float(((a-b)**2).sum()),4))

## 3. Approximate search (FAISS HNSW) + recall vs the exact baseline
If faiss isn't installed we simulate an approximate index by scanning a random subset,
so the recall concept still runs.

In [ ]:
def approx_ids(query, matrix, k=10):
    try:
        import faiss
        idx = faiss.IndexHNSWFlat(matrix.shape[1], 32)
        idx.hnsw.efSearch = 64
        idx.add(matrix)
        _, ids = idx.search(query.reshape(1,-1), k)
        return ids[0]
    except Exception as e:
        # fallback: search only 30% of vectors (mimics an approximate index)
        sub = rng.choice(len(matrix), size=int(0.3*len(matrix)), replace=False)
        local = flat_topk(query, matrix[sub], k)
        return sub[local]

approx = set(approx_ids(q, corpus, 10).tolist())
recall = len(set(exact.tolist()) & approx) / len(exact)
print(f"recall@10 (approx vs exact): {recall:.2f}")
print("higher efSearch (or a real ANN index) -> higher recall, more latency")

## 4. Metadata-filtered search (access control / recency)
Real queries are 'similar AND allowed'. Pre-filter restricts candidates before ranking.

In [ ]:
import operator
docs = [
  {"text":"Acme Q1 2024 minutes","tenant":"acme","year":2024},
  {"text":"Acme Q4 2022 minutes","tenant":"acme","year":2022},
  {"text":"Globex memo 2024","tenant":"globex","year":2024},
  {"text":"Acme HR handbook 2023","tenant":"acme","year":2023},
]
mvecs = rng.randn(len(docs), 32); mvecs /= np.linalg.norm(mvecs,axis=1,keepdims=True)
qv = mvecs[0] + 0.1*rng.randn(32); qv /= np.linalg.norm(qv)

OPS = {"$gte":operator.ge,"$gt":operator.gt,"$lte":operator.le,"$lt":operator.lt}
def keep(meta, where):
    for f, cond in where.items():
        val = meta.get(f)
        if isinstance(cond, dict):
            if not all(OPS[o](val, t) for o,t in cond.items()): return False
        elif val != cond: return False
    return True

def filtered_search(qv, where, k=3):
    cand = [(i,d) for i,d in enumerate(docs) if keep(d, where)]   # PRE-filter
    ranked = sorted(cand, key=lambda x: -float(mvecs[x[0]] @ qv))
    return [(d["text"], round(float(mvecs[i] @ qv),3)) for i,d in ranked[:k]]

print("tenant=acme, year>=2023 only:")
for t,s in filtered_search(qv, {"tenant":"acme","year":{"$gte":2023}}):
    print(f"  {s:>6}  {t}")

> Note only Acme's 2023+ docs are eligible — Globex vectors are never even
scored. That is tenant isolation implemented as a pre-filter.

## 5. Exercises
1. Lower the approximate fraction in §3 (0.3 -> 0.1). What happens to recall@10?
2. Install `faiss-cpu` and compare real HNSW recall at efSearch 16 vs 128.
3. Switch §4 to post-filtering (rank first, then drop) and force a case that returns < k.
4. Add an ACL list to each doc's metadata and filter by 'current_user in acl'.
5. Time exact vs approximate search as N grows to 200k.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Vector Search`
- Snippets: `04 Code Snippets/LLM/Flat vs FAISS Vector Search`, `.../Filtered Vector Search with Metadata`
- MOC: `06 Maps of Content/LLM Engineering Concepts`